# 🏛️ LutherICPU — 1-Click Cloud GPU 3D Gaussian Splatting Trainer
**Train photorealistic 3D Gaussian Splatting models from raw photographs in ~12–15 minutes on a free Google Colab GPU.**

---
### 🚀 How to Use:
1. In the top menu, go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU**.
2. Run the cells below in order.
3. Upload your `images.zip` (containing your multi-angle source photos).
4. When training finishes, `luther_simulation_bundle.zip` will automatically download to your computer.
5. Place it in your project or run `python simulate.py -b luther_simulation_bundle.zip` to enjoy 60 FPS interactive 3D simulation on your local CPU!

In [ ]:
# Step 1: Install CUDA-accelerated 3D Gaussian Splatting & COLMAP
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq colmap ffmpeg
!git clone --recursive https://github.com/camenduru/gaussian-splatting /content/gaussian-splatting
%cd /content/gaussian-splatting
!pip install -q plyfile tqdm scipy trimesh submodules/diff-gaussian-rasterization submodules/simple-knn

In [ ]:
# Step 2: Upload your Photographs (Zip file)
import os, zipfile, shutil, json
from google.colab import files

workspace_dir = "/content/scene_data"
if os.path.exists(workspace_dir):
    shutil.rmtree(workspace_dir)
os.makedirs(os.path.join(workspace_dir, "input"), exist_ok=True)

print("[*] Please upload your images zip file (e.g. photos.zip):")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(os.path.join(workspace_dir, "input"))

# Flatten nested directories if any
input_path = os.path.join(workspace_dir, "input")
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".PNG", ".JPEG")
all_files = []
for root, dirs, fnames in os.walk(input_path):
    for f in fnames:
        if f.endswith(valid_exts):
            all_files.append(os.path.join(root, f))

target_input = os.path.join(workspace_dir, "images")
os.makedirs(target_input, exist_ok=True)
for f in all_files:
    shutil.copy2(f, os.path.join(target_input, os.path.basename(f)))

print(f"[SUCCESS] Extracted {len(all_files)} source photographs ready for calibration!")

In [ ]:
# Step 3: Run Fast GPU COLMAP Structure-from-Motion
%cd /content/gaussian-splatting
!python convert.py -s /content/scene_data --camera PINHOLE

In [ ]:
# Step 4: Train 3D Gaussian Splatting (30,000 Gradient Descent Iterations)
%cd /content/gaussian-splatting
!python train.py -s /content/scene_data -m /content/scene_data/output_model --iterations 30000

In [ ]:
# Step 5: Convert to Direct GPU-Ready .splat Binary Buffer & Bundle for Local 60 FPS Viewport
import struct
import numpy as np
from plyfile import PlyData

ply_path = "/content/scene_data/output_model/point_cloud/iteration_30000/point_cloud.ply"
splat_path = "/content/scene.splat"

print("[*] Converting optimized 3DGS PLY to compact WebGL binary .splat buffer...")
ply = PlyData.read(ply_path)
v = ply['vertex']
count = len(v)

with open(splat_path, 'wb') as f:
    for i in range(count):
        # Position (x, y, z float32)
        f.write(struct.pack('fff', float(v['x'][i]), float(v['y'][i]), float(v['z'][i])))
        # Scale (sx, sy, sz float32 exp-mapped)
        sx = float(np.exp(v['scale_0'][i]))
        sy = float(np.exp(v['scale_1'][i]))
        sz = float(np.exp(v['scale_2'][i]))
        f.write(struct.pack('fff', sx, sy, sz))
        # Color (rgba uint8 premultiplied)
        SH_C0 = 0.28209479177387814
        r = int(np.clip((v['f_dc_0'][i] * SH_C0 + 0.5) * 255.0, 0, 255))
        g = int(np.clip((v['f_dc_1'][i] * SH_C0 + 0.5) * 255.0, 0, 255))
        b = int(np.clip((v['f_dc_2'][i] * SH_C0 + 0.5) * 255.0, 0, 255))
        op = float(v['opacity'][i])
        a = int(np.clip((1.0 / (1.0 + np.exp(-op))) * 255.0, 0, 255))
        f.write(struct.pack('BBBB', r, g, b, a))
        # Rotation (qw, qx, qy, qz float32)
        f.write(struct.pack('ffff', float(v['rot_0'][i]), float(v['rot_1'][i]), float(v['rot_2'][i]), float(v['rot_3'][i])))

# Package simulation bundle
bundle_zip = "/content/luther_simulation_bundle.zip"
with zipfile.ZipFile(bundle_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(splat_path, "scene.splat")
    zf.write(ply_path, "point_cloud.ply")
    # Write manifest
    manifest = {
        "scene_name": "colab_trained_scene",
        "splat_count": count,
        "iterations": 30000,
        "hardware": "Cloud T4 GPU"
    }
    zf.writestr("manifest.json", json.dumps(manifest, indent=2))

print(f"[SUCCESS] Simulation Bundle Ready ({os.path.getsize(bundle_zip) / (1024*1024):.1f} MB)!")
print("[*] Downloading luther_simulation_bundle.zip to your computer...")
files.download(bundle_zip)